# Musicm8 — AI Producer + Neural Vocals

This is the main one-click workflow:

**reference songs → stems/MIDI/tokens/spectral fingerprints → producer AI → original lyrics → chord-aware composition → inverse synth matching → polished instrumental → lyric-to-melody vocal score → ACE-Step neural singing → vocal FX/doubles → final song**

Change `IDEA` and `VOCAL_STYLE`, then run the big cell. The first vocal run is much larger/slower because ACE-Step 1.5 and its base model are installed in an isolated Python 3.12 environment and cached in Drive. Later runs reuse the model weights.

The normal Colab runtime can stay on its current Python version; the singing backend is isolated automatically.


In [ ]:
# ============================================================
# MUSICM8 — ONE CLICK COMPLETE SONG
# ============================================================

import os, sys, json, shutil, subprocess
from pathlib import Path

# ------------------------------------------------------------
# EDIT THESE
# ------------------------------------------------------------
IDEA = "dark UK garage song about knowing a relationship is over but not being able to leave, emotional chords, deep moving bass"
BARS = 32
SEED = 42

VOCALS = True
VOCAL_STYLE = "expressive contemporary lead vocal, intimate verses, emotional hook, clear lyrics, modern UK electronic production"
VOCAL_LANGUAGE = "en"
VOCAL_STEPS = 40

MATCH_ITERS = 48
MATCH_SECONDS = 3.0
FORCE_SOUND_MATCH = False
AI_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

ROOT = Path("/content/drive/MyDrive/Musicm8")
AUDIO = ROOT / "audio"
WORK = ROOT / "work"
REPO = Path("/content/Musicm8")
REPO_URL = "https://github.com/Elephant-logic/Musicm8.git"
AUDIO.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)

# Persistent model/download caches.
os.environ["HF_HOME"] = str(WORK / "hf_cache")
os.environ["UV_CACHE_DIR"] = str(WORK / "uv_cache")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Pull latest Musicm8 without restarting the GPU session.
if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "--hard", "origin/main"], check=True)
else:
    shutil.rmtree(REPO, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

os.chdir(REPO)
print("Installing Musicm8 dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-ai.txt"], check=True)
# Optional IPA phoneme guide for vocal_score.json.
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "espeak-ng"], check=False)

import torch
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU connected. Runtime → Change runtime type → GPU, then run THIS SAME CELL again.")
print("GPU:", torch.cuda.get_device_name(0))

audio_files = [p for p in AUDIO.rglob("*") if p.is_file() and p.suffix.lower() in {".wav",".mp3",".flac",".m4a",".aac",".ogg",".opus"}]
print("Reference songs:", len(audio_files))
if not audio_files:
    raise FileNotFoundError(f"Put your reference songs in {AUDIO}")

cmd = [
    sys.executable, "-u", "ai_producer_workflow.py",
    "--root", str(ROOT),
    "--repo", str(REPO),
    "--idea", IDEA,
    "--bars", str(BARS),
    "--seed", str(SEED),
    "--ai-model", AI_MODEL,
    "--match-iters", str(MATCH_ITERS),
    "--match-seconds", str(MATCH_SECONDS),
    "--vocal-style", VOCAL_STYLE,
    "--vocal-language", VOCAL_LANGUAGE,
    "--vocal-steps", str(VOCAL_STEPS),
]
if not VOCALS:
    cmd.append("--no-vocals")
if FORCE_SOUND_MATCH:
    cmd.append("--force-sound-match")

print("\n🚀 Starting Musicm8 complete-song workflow...")
subprocess.run(cmd, check=True)

PROJECT = WORK / "ai_projects/latest"
MASTER = PROJECT / "master.wav"
INSTRUMENTAL = PROJECT / "master_instrumental.wav"
LYRICS = PROJECT / "lyrics.txt"
VOCAL = PROJECT / "vocals/vocal_mix.wav"
RAW_VOCAL = PROJECT / "vocals/neural_lead_raw.wav"

print("\n✅ MUSICM8 COMPLETE SONG PROJECT")
print("Idea        :", IDEA)
print("Lyrics      :", LYRICS)
print("Vocal score :", PROJECT / "vocal_score.json")
print("MIDI        :", PROJECT / "arrangement.mid")
print("MIDI stems  :", PROJECT / "midi_stems")
print("Instruments :", PROJECT / "audio_stems_polished")
print("Vocals      :", PROJECT / "vocals")
print("Final master:", MASTER)

if LYRICS.exists():
    print("\n📝 LYRICS\n")
    print(LYRICS.read_text(encoding="utf-8"))

from IPython.display import Audio, display
if INSTRUMENTAL.exists():
    print("\n🎹 INSTRUMENTAL")
    display(Audio(str(INSTRUMENTAL)))
if RAW_VOCAL.exists():
    print("\n🎤 RAW NEURAL VOCAL")
    display(Audio(str(RAW_VOCAL)))
if VOCAL.exists():
    print("\n🎚️ PROCESSED VOCAL MIX")
    display(Audio(str(VOCAL)))
if MASTER.exists():
    print("\n🎵 MUSICM8 FINAL SONG")
    display(Audio(str(MASTER)))


## Optional: inspect saved files


In [ ]:
from pathlib import Path
root = Path("/content/drive/MyDrive/Musicm8/work")
project = root / "ai_projects/latest"
print("ACE-Step models:", root / "ace_step_models")
print("Latest project :", project)
for p in sorted(project.rglob("*")) if project.exists() else []:
    if p.is_file(): print(" -", p)
